# FeatureImpact — Notebook 7: Auto Report Generation

**Objective:** Automatically compile all experiment results into a clean, shareable HTML report that non-technical stakeholders (product managers, designers, leadership) can read without touching the notebooks.

This is the final deliverable of the FeatureImpact pipeline:
- Runs all tests end-to-end
- Generates a colour-coded results table
- Embeds charts inline
- Writes a plain-English recommendation section
- Exports to a self-contained `experiment_report.html`

## 1. Re-run All Tests (Single Cell Pipeline)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import base64
import io
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from datetime import datetime

matplotlib.use("Agg")  # Non-interactive backend for report generation
np.random.seed(42)

# ─── Load data ───────────────────────────────────────────────────────────────
df = pd.read_csv("data/ab_experiment_raw.csv", parse_dates=["entry_date"])
df_clean = df.dropna(subset=["session_duration_sec"]).copy()

ctrl     = df[df["group"] == "control"]
trt      = df[df["group"] == "treatment"]
ctrl_c   = df_clean[df_clean["group"] == "control"]
trt_c    = df_clean[df_clean["group"] == "treatment"]

n_ctrl, n_trt = len(ctrl), len(trt)

# ─── Conversion rate test ────────────────────────────────────────────────────
conv_ctrl, conv_trt = ctrl["converted"].sum(), trt["converted"].sum()
rate_ctrl, rate_trt = conv_ctrl/n_ctrl, conv_trt/n_trt
z_conv, p_conv = proportions_ztest([conv_trt, conv_ctrl], [n_trt, n_ctrl])
lift_conv = rate_trt - rate_ctrl

# ─── Session duration test ───────────────────────────────────────────────────
t_sess, p_sess = stats.ttest_ind(trt_c["session_duration_sec"], ctrl_c["session_duration_sec"], equal_var=False)
lift_sess = trt_c["session_duration_sec"].mean() - ctrl_c["session_duration_sec"].mean()

# ─── CTR test ────────────────────────────────────────────────────────────────
ctr_ctrl, ctr_trt = ctrl["clicked_prompt"].sum(), trt["clicked_prompt"].sum()
rate_ctr_ctrl, rate_ctr_trt = ctr_ctrl/n_ctrl, ctr_trt/n_trt
contingency = np.array([[ctr_trt, n_trt - ctr_trt], [ctr_ctrl, n_ctrl - ctr_ctrl]])
chi2_ctr, p_ctr, _, _ = stats.chi2_contingency(contingency)
lift_ctr = rate_ctr_trt - rate_ctr_ctrl

# ─── Multiple comparisons correction ────────────────────────────────────────
raw_pvals = [p_conv, p_sess, p_ctr]
reject, pvals_bh, _, _ = multipletests(raw_pvals, alpha=0.05, method="fdr_bh")

# ─── Bayesian: P(TRT > CTRL) ─────────────────────────────────────────────────
post_ctrl = stats.beta(1 + conv_ctrl, 1 + n_ctrl - conv_ctrl)
post_trt  = stats.beta(1 + conv_trt,  1 + n_trt  - conv_trt)
samples_c = post_ctrl.rvs(100_000)
samples_t = post_trt.rvs(100_000)
p_trt_wins = np.mean(samples_t > samples_c)

print("All tests complete. Building report...")

## 2. Generate Charts for the Report

In [ ]:
def fig_to_base64(fig):
    """Convert a matplotlib figure to base64 PNG for embedding in HTML."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return encoded

palette = {"control": "#4C9BE8", "treatment": "#F4845F"}

# Chart 1: Metric comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Experiment Results: Control vs Treatment", fontsize=13, fontweight="bold", y=1.02)

metric_data = [
    ("Conversion Rate",    [rate_ctrl,              rate_trt             ], ".1%"),
    ("Avg Session (sec)",  [ctrl_c["session_duration_sec"].mean(), trt_c["session_duration_sec"].mean()], ".0f"),
    ("Click-Through Rate", [rate_ctr_ctrl,           rate_ctr_trt         ], ".1%"),
]

for ax, (title, values, fmt) in zip(axes, metric_data):
    colors = [palette["control"], palette["treatment"]]
    bars = ax.bar(["Control", "Treatment"], values, color=colors, edgecolor="white", width=0.5)
    ax.set_title(title, fontweight="bold")
    for bar, v in zip(bars, values):
        label = format(v, fmt) if fmt != ".0f" else f"{v:.1f}s"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                label, ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, max(values) * 1.25)

chart1_b64 = fig_to_base64(fig)

# Chart 2: p-value summary
fig2, ax2 = plt.subplots(figsize=(8, 4))
metrics_labels = ["Conversion Rate", "Session Duration", "CTR"]
bar_colors = ["#2ECC71" if r else "#E74C3C" for r in reject]
ax2.barh(metrics_labels, pvals_bh, color=bar_colors, edgecolor="white")
ax2.axvline(0.05, color="black", linestyle="--", linewidth=1.5, label="α = 0.05")
ax2.set_xlabel("BH-Corrected p-value")
ax2.set_title("Statistical Significance (BH-corrected p-values)", fontweight="bold")
ax2.legend()
for i, (p, label) in enumerate(zip(pvals_bh, metrics_labels)):
    ax2.text(p + 0.001, i, f"{p:.4f}", va="center", fontsize=9)
chart2_b64 = fig_to_base64(fig2)

# Chart 3: Bayesian posterior
fig3, ax3 = plt.subplots(figsize=(8, 4))
x = np.linspace(0.05, 0.22, 1000)
ax3.fill_between(x, post_ctrl.pdf(x), alpha=0.35, color="#4C9BE8")
ax3.plot(x, post_ctrl.pdf(x), color="#4C9BE8", linewidth=2.5, label=f"Control (mean={post_ctrl.mean():.3f})")
ax3.fill_between(x, post_trt.pdf(x), alpha=0.35, color="#F4845F")
ax3.plot(x, post_trt.pdf(x), color="#F4845F", linewidth=2.5, label=f"Treatment (mean={post_trt.mean():.3f})")
ax3.set_title(f"Bayesian Posteriors  |  P(Treatment > Control) = {p_trt_wins:.1%}", fontweight="bold")
ax3.set_xlabel("Conversion Rate")
ax3.set_ylabel("Density")
ax3.legend()
chart3_b64 = fig_to_base64(fig3)

print("Charts generated.")

## 3. Assemble and Export HTML Report

In [ ]:
def sig_badge(is_significant):
    if is_significant:
        return '<span style="background:#2ECC71;color:white;padding:2px 8px;border-radius:12px;font-size:12px;">✅ Significant</span>'
    return '<span style="background:#E74C3C;color:white;padding:2px 8px;border-radius:12px;font-size:12px;">❌ Not Significant</span>'


report_date = datetime.now().strftime("%B %d, %Y at %H:%M")

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>FeatureImpact — Experiment Report</title>
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 960px; margin: 0 auto; padding: 40px 24px;
            color: #1a1a2e; background: #f8f9fa; }}
    h1   {{ color: #0747A6; border-bottom: 3px solid #0747A6; padding-bottom: 12px; }}
    h2   {{ color: #0747A6; margin-top: 40px; }}
    .meta {{ background: #fff; border-left: 4px solid #0747A6;
             padding: 16px 20px; border-radius: 4px; margin-bottom: 32px;
             font-size: 14px; color: #555; }}
    table {{ width: 100%; border-collapse: collapse; background: white;
             box-shadow: 0 1px 4px rgba(0,0,0,0.08); border-radius: 8px; overflow: hidden; }}
    th {{ background: #0747A6; color: white; padding: 12px 16px; text-align: left; font-size: 13px; }}
    td {{ padding: 11px 16px; border-bottom: 1px solid #f0f0f0; font-size: 13px; }}
    tr:last-child td {{ border-bottom: none; }}
    tr:hover td {{ background: #f8fbff; }}
    .chart-box {{ background: white; padding: 20px; border-radius: 8px;
                  box-shadow: 0 1px 4px rgba(0,0,0,0.08); margin: 24px 0; }}
    .chart-box img {{ width: 100%; }}
    .rec-box {{ background: #E6F4FF; border-left: 4px solid #0747A6;
                padding: 20px 24px; border-radius: 4px; margin: 24px 0; }}
    .rec-box h3 {{ margin-top: 0; color: #0747A6; }}
    .kpi-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin: 24px 0; }}
    .kpi {{ background: white; padding: 16px; border-radius: 8px; text-align: center;
             box-shadow: 0 1px 4px rgba(0,0,0,0.08); }}
    .kpi .val {{ font-size: 28px; font-weight: 700; color: #0747A6; }}
    .kpi .label {{ font-size: 12px; color: #888; margin-top: 4px; }}
    footer {{ text-align: center; color: #aaa; font-size: 12px; margin-top: 60px; }}
  </style>
</head>
<body>

<h1>FeatureImpact — Experiment Report</h1>
<div class="meta">
  <strong>Experiment:</strong> jira_dashboard_v2_rollout &nbsp;|&nbsp;
  <strong>Duration:</strong> 14 days &nbsp;|&nbsp;
  <strong>Total users:</strong> {n_ctrl + n_trt:,} &nbsp;|&nbsp;
  <strong>Generated:</strong> {report_date}
</div>

<h2>Key Metrics at a Glance</h2>
<div class="kpi-grid">
  <div class="kpi"><div class="val">{n_ctrl + n_trt:,}</div><div class="label">Total users</div></div>
  <div class="kpi"><div class="val">{lift_conv:+.1%}</div><div class="label">Conversion lift</div></div>
  <div class="kpi"><div class="val">{lift_sess:+.0f}s</div><div class="label">Session duration lift</div></div>
  <div class="kpi"><div class="val">{p_trt_wins:.0%}</div><div class="label">P(Treatment wins)</div></div>
</div>

<h2>Metric Comparison</h2>
<div class="chart-box"><img src="data:image/png;base64,{chart1_b64}" alt="Metric comparison chart"></div>

<h2>Statistical Test Results</h2>
<table>
  <thead>
    <tr>
      <th>Metric</th>
      <th>Control</th>
      <th>Treatment</th>
      <th>Absolute Lift</th>
      <th>Raw p-value</th>
      <th>BH-corrected p</th>
      <th>Result</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Conversion Rate</strong></td>
      <td>{rate_ctrl:.2%}</td>
      <td>{rate_trt:.2%}</td>
      <td>{lift_conv:+.2%}</td>
      <td>{p_conv:.6f}</td>
      <td>{pvals_bh[0]:.6f}</td>
      <td>{sig_badge(reject[0])}</td>
    </tr>
    <tr>
      <td><strong>Session Duration</strong></td>
      <td>{ctrl_c["session_duration_sec"].mean():.1f}s</td>
      <td>{trt_c["session_duration_sec"].mean():.1f}s</td>
      <td>{lift_sess:+.1f}s</td>
      <td>{p_sess:.6f}</td>
      <td>{pvals_bh[1]:.6f}</td>
      <td>{sig_badge(reject[1])}</td>
    </tr>
    <tr>
      <td><strong>Click-Through Rate</strong></td>
      <td>{rate_ctr_ctrl:.2%}</td>
      <td>{rate_ctr_trt:.2%}</td>
      <td>{lift_ctr:+.2%}</td>
      <td>{p_ctr:.6f}</td>
      <td>{pvals_bh[2]:.6f}</td>
      <td>{sig_badge(reject[2])}</td>
    </tr>
  </tbody>
</table>
<p style="font-size:12px;color:#888;">BH = Benjamini-Hochberg correction for multiple comparisons (FDR controlled at 5%).</p>

<h2>Statistical Significance Overview</h2>
<div class="chart-box"><img src="data:image/png;base64,{chart2_b64}" alt="p-value chart"></div>

<h2>Bayesian Analysis</h2>
<div class="chart-box"><img src="data:image/png;base64,{chart3_b64}" alt="Bayesian posteriors"></div>
<p style="font-size:13px;color:#555;">
  The Bayesian Beta-Binomial model estimates <strong>P(Treatment &gt; Control) = {p_trt_wins:.1%}</strong>.
  The treatment group's posterior mean conversion rate is <strong>{post_trt.mean():.3f}</strong>
  vs control's <strong>{post_ctrl.mean():.3f}</strong>.
</p>

<h2>Recommendation</h2>
<div class="rec-box">
  <h3>🚀 Recommended Action: Ship Treatment to All Users</h3>
  <p>
    The new Jira Dashboard feature demonstrates statistically significant improvements across
    all three primary metrics after Benjamini-Hochberg correction. The treatment group shows
    a <strong>{lift_conv:+.2%} absolute lift in conversion rate</strong>,
    <strong>{lift_sess:+.1f}s longer average session duration</strong>, and
    <strong>{lift_ctr:+.2%} improvement in click-through rate</strong>.
  </p>
  <p>
    The Bayesian analysis corroborates this: there is a <strong>{p_trt_wins:.0%} probability
    that the treatment conversion rate genuinely exceeds the control</strong>.
  </p>
  <p><strong>Next steps:</strong></p>
  <ul>
    <li>Proceed with full rollout to all users.</li>
    <li>Monitor conversion rate and session duration weekly for 30 days post-launch.</li>
    <li>Investigate the premium plan segment further — EDA showed the largest lift there.</li>
    <li>Consider a follow-up experiment to optimise the specific UI components driving engagement.</li>
  </ul>
</div>

<h2>Methodology Notes</h2>
<table>
  <thead><tr><th>Aspect</th><th>Choice</th><th>Rationale</th></tr></thead>
  <tbody>
    <tr><td>Significance level</td><td>α = 0.05</td><td>Standard industry threshold</td></tr>
    <tr><td>Conversion test</td><td>Two-proportion Z-test</td><td>Large sample, binary outcome</td></tr>
    <tr><td>Session duration test</td><td>Welch's t-test + Mann-Whitney U</td><td>Robustness check for non-normality</td></tr>
    <tr><td>Multiple comparisons</td><td>Benjamini-Hochberg (FDR)</td><td>Less conservative than Bonferroni for 3 metrics</td></tr>
    <tr><td>Bayesian model</td><td>Beta-Binomial</td><td>Conjugate prior; closed-form posterior</td></tr>
    <tr><td>SRM check</td><td>Chi-square on group sizes</td><td>Validates randomisation integrity</td></tr>
  </tbody>
</table>

<footer>Generated by FeatureImpact A/B Test Analysis Engine &nbsp;|&nbsp; Gagan R &nbsp;|&nbsp; {report_date}</footer>
</body>
</html>
"""

report_path = "experiment_report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Report saved to: {report_path}")
print(f"   File size: {len(html) / 1024:.1f} KB")
print("   Open in any browser — fully self-contained, no dependencies.")

## 4. Preview Summary Table

In [ ]:
summary = pd.DataFrame({
    "Metric":           ["Conversion Rate", "Session Duration", "CTR"],
    "Control":          [f"{rate_ctrl:.2%}", f"{ctrl_c['session_duration_sec'].mean():.1f}s", f"{rate_ctr_ctrl:.2%}"],
    "Treatment":        [f"{rate_trt:.2%}",  f"{trt_c['session_duration_sec'].mean():.1f}s",  f"{rate_ctr_trt:.2%}"],
    "Lift":             [f"{lift_conv:+.2%}", f"{lift_sess:+.1f}s", f"{lift_ctr:+.2%}"],
    "BH p-value":       [f"{p:.4f}" for p in pvals_bh],
    "Significant":      ["✅" if r else "❌" for r in reject],
})

print("=" * 80)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 80)
print(summary.to_string(index=False))
print(f"\nBayesian: P(Treatment > Control) = {p_trt_wins:.1%}")
print("\n✅ Report exported to experiment_report.html — ready to share with stakeholders.")

## Project Complete 🎉

| Notebook | What It Demonstrates |
|---|---|
| 1 — Data Generation | Data engineering, reproducibility, synthetic data design |
| 2 — EDA | Data quality, SRM checks, subgroup analysis |
| 3 — Frequentist Testing | Hypothesis testing, confidence intervals, effect sizes |
| 4 — Power Analysis | Experimental design, sample size calculation |
| 5 — Bayesian Testing | Probabilistic reasoning, business-friendly communication |
| 6 — Sequential Testing | Peeking problem, SPRT, multiple comparisons (BH) |
| 7 — Report Generation | Stakeholder communication, automated deliverables |

This project directly maps to Atlassian's internship responsibilities:
- ✅ Measure impact of product strategy via experiments
- ✅ Share high-quality insights with peers and leadership
- ✅ Translate business questions into technical solutions
- ✅ Collaborate cross-functionally (PM-ready HTML report)